# Palimpsest — OSCD Fine-tuning (Colab T4)

Fine-tunes ChangeFormer (EfficientNet-B2 Siamese) from a LEVIR-CD pre-trained
checkpoint on the **Onera Satellite Change Detection (OSCD)** dataset — real
Sentinel-2 at **10 m/px**, matching production resolution.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Edit the **Configuration** cell (Section 1)
3. `Runtime → Run all`

Expected wall time: **~2 h** on T4 for 150 epochs.

## 0 · Runtime check + install

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU — go to Runtime → Change runtime type → T4 GPU')
print(f'GPU: {r.stdout.strip()}')

In [ ]:
%%capture
!pip install -q timm rasterio google-cloud-storage kaggle

## 1 · Configuration — edit this cell

In [ ]:
# ════════════════════════════════════════════════════════
#  ★  EDIT THESE  ★
# ════════════════════════════════════════════════════════
GCS_BUCKET      = 'palimpsest-bucket'
GCP_PROJECT     = 'project-8c7ca821-aa7a-45ea-88b'

KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'
KAGGLE_KEY      = 'YOUR_KAGGLE_API_KEY'

GCS_CKPT_IN     = 'checkpoints/changeformer_levir.pth'
GCS_CKPT_OUT    = 'checkpoints/changeformer_oscd.pth'
# ════════════════════════════════════════════════════════

# Training hyper-parameters (safe defaults)
EPOCHS           = 150
BATCH_SIZE       = 8
PEAK_LR          = 5e-6
POS_WEIGHT       = 5.0
PATCH_SIZE       = 256
PATCHES_PER_CITY = 60
WARMUP_EPOCHS    = 5

print('Config loaded ✓')

## 2 · GCS authentication

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('GCS auth OK ✓')

## 3 · Kaggle auth + OSCD download

In [ ]:
import os, json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle auth configured ✓')

In [ ]:
OSCD_SLUG    = 'soumikrakshit/onera-satellite-change-detection-dataset'
OSCD_RAW_DIR = '/content/oscd_raw'

if not os.path.exists(OSCD_RAW_DIR) or not os.listdir(OSCD_RAW_DIR):
    print('Downloading OSCD from Kaggle (~1.5 GB) …')
    os.makedirs(OSCD_RAW_DIR, exist_ok=True)
    !kaggle datasets download -d {OSCD_SLUG} -p {OSCD_RAW_DIR} --unzip
else:
    print('OSCD already present.')
!ls -1 {OSCD_RAW_DIR}

## 4 · Locate dataset directories (handles nested Kaggle layout)

In [ ]:
from pathlib import Path

def _find_dir(root: Path, keywords: list) -> Path | None:
    """First subdirectory whose name contains any keyword (case-insensitive)."""
    for p in sorted(root.iterdir()):
        if p.is_dir() and any(k.lower() in p.name.lower() for k in keywords):
            return p
    return None

def _dig_if_single(d: Path) -> Path:
    """
    Kaggle packs the dataset as:
      oscd_raw/
        images/
          Onera Satellite Change Detection dataset - Images/   ← single subdir
            abudhabi/
    This function dives one level when there is exactly one subdirectory.
    """
    subs = [p for p in d.iterdir() if p.is_dir()]
    return subs[0] if len(subs) == 1 else d

oscd_root  = Path(OSCD_RAW_DIR)
images_dir = _find_dir(oscd_root, ['images', 'image'])
labels_dir = _find_dir(oscd_root, ['train_labels', 'train labels', 'trainlabels', 'labels'])

# Try one level deeper if not found at root
if images_dir is None or labels_dir is None:
    for sub in sorted(oscd_root.iterdir()):
        if sub.is_dir():
            images_dir = images_dir or _find_dir(sub, ['images'])
            labels_dir = labels_dir or _find_dir(sub, ['train_labels', 'train labels', 'labels'])

assert images_dir, f'Cannot find images dir under {oscd_root}'
assert labels_dir, f'Cannot find labels dir under {oscd_root}'

# Dive past the 'Onera Satellite ... - Images' intermediate directory
images_dir = _dig_if_single(images_dir)
labels_dir = _dig_if_single(labels_dir)

cities_found = sorted(p.name for p in images_dir.iterdir() if p.is_dir())
print(f'images_dir : {images_dir}')
print(f'labels_dir : {labels_dir}')
print(f'Cities in images ({len(cities_found)}): {cities_found}')

## 5 · Mask diagnostic — filter out cities with degenerate labels

The Kaggle version of OSCD includes **test-set cities** (e.g. `pisa`, `brasilia`)
that have placeholder/inverted change masks (100 % change or 0 % change).
Training on these would teach the model a trivial solution (predict everything
as changed → F1≈1.0 but meaningless).  This cell:

1. Reads every available mask and prints the per-city change fraction.
2. Keeps only cities with **0.3 % – 85 %** change (physically plausible range).

In [ ]:
import rasterio
import numpy as np

_T1_OPTS = ['imgs_1_rect', 'imgs_1']
_T2_OPTS = ['imgs_2_rect', 'imgs_2']

def _find_sub(city_dir, opts):
    for n in opts:
        p = city_dir / n
        if p.is_dir(): return p
    return None

def _mask_change_pct(mask_path: Path) -> float:
    """
    Retorna el porcentaje de píxeles de cambio.
    Auto-detecta la convención del archivo probando múltiples interpretaciones:
      cm > 1  → este dataset Kaggle: 1=no-change, 2=change
      cm > 0  → convención estándar ONERA: 0=no-change, 1=change
      cm == 0 → convención invertida: 0=change
      cm > 127 → escala de grises: 255=change
    Devuelve la primera que produzca una fracción plausible (0.3-85%).
    """
    with rasterio.open(mask_path) as src:
        cm = src.read(1).astype('float32')
    for mask in [cm > 1, cm > 0, cm == 0, cm > 127]:
        pct = float(mask.mean() * 100)
        if 0.3 <= pct <= 85.0:
            return pct
    # Ninguna convención dio resultado plausible — devolver el valor bruto para diagnóstico
    return float((cm > 0).mean() * 100)

print(f'Scanning {len(cities_found)} cities …\n')
print(f'{"City":<22} {"Images":>8} {"Labels":>8} {"change%":>9} {"verdict"}')
print('─' * 60)

GOOD_CITIES     = []
SKIP_NO_LABEL   = []
SKIP_DEGENERATE = []

for city in cities_found:
    cd = images_dir / city
    t1 = _find_sub(cd, _T1_OPTS)
    t2 = _find_sub(cd, _T2_OPTS)
    has_images = (t1 is not None and t2 is not None)

    mask_dir  = labels_dir / city / 'cm'
    mask_path = mask_dir / f'{city}-cm.tif'
    if not mask_path.exists(): mask_path = mask_dir / 'cm.tif'
    has_label = mask_path.exists()

    if not has_images or not has_label:
        SKIP_NO_LABEL.append(city)
        print(f'{city:<22} {str(has_images):>8} {str(has_label):>8} {"—":>9}  skip (no img or label)')
        continue

    pct = _mask_change_pct(mask_path)

    if 0.3 <= pct <= 85.0:
        GOOD_CITIES.append(city)
        print(f'{city:<22} {"✓":>8} {"✓":>8} {pct:>8.2f}%  ✓ valid')
    else:
        SKIP_DEGENERATE.append(city)
        tag = 'all-change (placeholder)' if pct > 85 else 'no change'
        print(f'{city:<22} {"✓":>8} {"✓":>8} {pct:>8.2f}%  ✗ REJECTED — {tag}')

print('─' * 60)
print(f'Good: {len(GOOD_CITIES)}  |  Skipped (no label): {len(SKIP_NO_LABEL)}  |  Skipped (degenerate): {len(SKIP_DEGENERATE)}')
print(f'\nGood cities: {GOOD_CITIES}')

if len(GOOD_CITIES) < 4:
    raise RuntimeError(
        f'Only {len(GOOD_CITIES)} valid cities found — not enough to train.\n'
        'Check that images_dir and labels_dir are correct.')

## 6 · Download LEVIR-CD checkpoint from GCS

In [ ]:
CKPT_LEVIR = '/content/checkpoints/changeformer_levir.pth'
CKPT_OSCD  = '/content/checkpoints/changeformer_oscd.pth'
os.makedirs('/content/checkpoints', exist_ok=True)

if not os.path.exists(CKPT_LEVIR):
    print(f'Downloading from gs://{GCS_BUCKET}/{GCS_CKPT_IN} …')
    !gcloud storage cp gs://{GCS_BUCKET}/{GCS_CKPT_IN} {CKPT_LEVIR}

print(f'Checkpoint: {os.path.getsize(CKPT_LEVIR)/1e6:.1f} MB  ✓')

from torch.utils.data import Dataset, DataLoader

_BANDS = ['B02', 'B03', 'B04', 'B08']   # Blue, Green, Red, NIR  (10 m/px)


def _load_bands(img_dir: Path, bands: list) -> np.ndarray:
    """
    Load per-band GeoTIFFs → (H, W, C) float32.
    Supports:
      • Exact   : B02.tif
      • Prefixed: S2A_OPER_…_B02.tif  (Kaggle raw format)
    """
    arrs = []
    for b in bands:
        tif = img_dir / f'{b}.tif'
        if not tif.exists():
            matches = sorted(img_dir.glob(f'*_{b}.tif'))
            if not matches:
                avail = sorted(p.name for p in img_dir.glob('*.tif'))[:5]
                raise FileNotFoundError(
                    f'No file matching {b}.tif or *_{b}.tif in {img_dir}\n'
                    f'  Available: {avail}')
            tif = matches[0]
        with rasterio.open(tif) as src:
            arrs.append(src.read(1).astype(np.float32))
    return np.stack(arrs, axis=-1)


def _load_mask(path: Path) -> np.ndarray:
    """
    Load OSCD change mask → (H, W) float32 binary {0=no-change, 1=change}.

    This Kaggle dataset uses a 3-value convention:
      1 = no-change  (background, ~96-99 % of pixels)
      2 = change     (what we want to detect,  ~1-4 %)
    So the correct binarisation is  cm == 2  (equivalently  cm > 1).

    The auto-detect fallback also handles the standard {0,1} and {0,255}
    conventions used by other OSCD re-packagings.
    """
    with rasterio.open(path) as src:
        cm = src.read(1).astype(np.float32)

    # Try each convention in order; return the first that gives a
    # physically plausible change fraction (0.3 % – 85 %).
    candidates = [
        ('cm > 1',   (cm > 1)),    # ← this dataset: 1=no-change, 2=change
        ('cm > 0',   (cm > 0)),    # standard: 0=no-change, 1 or 255=change
        ('cm == 0',  (cm == 0)),   # inverted: 0=change, positive=no-change
        ('cm > 127', (cm > 127)),  # RGB/grayscale: 255=change
    ]
    for name, mask_bool in candidates:
        pct = mask_bool.mean() * 100
        if 0.3 <= pct <= 85.0:
            return mask_bool.astype(np.float32)

    # Nothing worked — emit a clear diagnostic so the user can report it
    uv, uc = np.unique(cm, return_counts=True)
    diag = ', '.join(f'{int(v)}×{c}' for v, c in zip(uv[:8], uc[:8]))
    raise ValueError(
        f'Cannot determine mask convention for {path}\n'
        f'  Unique values: {diag}\n'
        f'  dtype={cm.dtype}  shape={cm.shape}')


def _norm_pair(before, after):
    """Joint percentile normalisation [p2, p98] → [0,1], per channel."""
    out_b = np.empty_like(before, dtype=np.float32)
    out_a = np.empty_like(after,  dtype=np.float32)
    for c in range(before.shape[-1]):
        b_ch, a_ch = before[..., c], after[..., c]
        joint = np.concatenate([b_ch.ravel(), a_ch.ravel()])
        lo, hi = np.percentile(joint, 2), np.percentile(joint, 98)
        d = hi - lo
        if d < 1e-8:
            out_b[..., c] = out_a[..., c] = 0.0
        else:
            out_b[..., c] = np.clip((b_ch - lo) / d, 0, 1)
            out_a[..., c] = np.clip((a_ch - lo) / d, 0, 1)
    return out_b, out_a


class OSCDDataset(Dataset):
    """
    Random-crop dataset for OSCD Sentinel-2 city pairs.
    Full images are cached in RAM on first access (~5 MB / city pair).
    """
    def __init__(self, images_dir, labels_dir, cities,
                 patch_size=256, patches_per_city=50, augment=True):
        self.ps, self.ppc, self.aug = patch_size, patches_per_city, augment
        images_dir, labels_dir = Path(images_dir), Path(labels_dir)
        self._triples = []
        for city in cities:
            cd = images_dir / city
            if not cd.is_dir(): continue
            t1 = _find_sub(cd, _T1_OPTS)
            t2 = _find_sub(cd, _T2_OPTS)
            if t1 is None or t2 is None: continue
            md = labels_dir / city / 'cm'
            mp = md / f'{city}-cm.tif'
            if not mp.exists(): mp = md / 'cm.tif'
            if not mp.exists(): continue
            self._triples.append((t1, t2, mp))
        if not self._triples:
            raise ValueError(f'No valid city pairs found.\n'
                             f'  images_dir={images_dir}\n  cities={cities}')
        print(f'[OSCDDataset] {len(self._triples)} cities  '
              f'ps={patch_size}  ppc={patches_per_city}  '
              f'aug={augment}  → {len(self)} samples/epoch')
        self._cache = {}

    def _city(self, i):
        if i not in self._cache:
            t1, t2, mp = self._triples[i]
            b, a = _norm_pair(_load_bands(t1, _BANDS), _load_bands(t2, _BANDS))
            self._cache[i] = (b, a, _load_mask(mp))
        return self._cache[i]

    def preload_all(self):
        print('Preloading:', end=' ', flush=True)
        for i in range(len(self._triples)):
            self._city(i)
            print(self._triples[i][0].parent.name, end=' ', flush=True)
        print('✓')

    def __len__(self): return len(self._triples) * self.ppc

    def __getitem__(self, idx):
        before, after, mask = self._city(idx % len(self._triples))
        H, W = before.shape[:2]
        ps   = self.ps
        if H < ps or W < ps:
            ph = max(0, ps - H); pw = max(0, ps - W)
            before = np.pad(before, ((0,ph),(0,pw),(0,0)), 'reflect')
            after  = np.pad(after,  ((0,ph),(0,pw),(0,0)), 'reflect')
            mask   = np.pad(mask,   ((0,ph),(0,pw)),       'reflect')
            H, W   = before.shape[:2]
        i = np.random.randint(0, H - ps + 1)
        j = np.random.randint(0, W - ps + 1)
        b = before[i:i+ps, j:j+ps].copy()
        a = after [i:i+ps, j:j+ps].copy()
        m = mask  [i:i+ps, j:j+ps].copy()
        if self.aug:
            if np.random.rand() > .5:
                b,a,m = b[:,::-1].copy(), a[:,::-1].copy(), m[:,::-1].copy()
            if np.random.rand() > .5:
                b,a,m = b[::-1].copy(), a[::-1].copy(), m[::-1].copy()
            k = np.random.randint(0, 4)
            if k:
                b,a,m = np.rot90(b,k).copy(), np.rot90(a,k).copy(), np.rot90(m,k).copy()
        return (torch.from_numpy(b.transpose(2,0,1)),
                torch.from_numpy(a.transpose(2,0,1)),
                torch.from_numpy(m).unsqueeze(0).float())


print('OSCDDataset defined ✓')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

_EFF_CHANNELS = [24, 48, 120, 352]
_EFF_OUT_IDX  = (1, 2, 3, 4)


class _ConvBnRelu(nn.Sequential):
    def __init__(self, in_ch, out_ch, k=1, p=0):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, k, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True))


class ChangeFormer(nn.Module):
    """Siamese EfficientNet-B2 change detection model. 4-channel input (BGRNIR)."""
    def __init__(self, in_channels=4, embed_dim=256):
        super().__init__()
        self.encoder = timm.create_model(
            'efficientnet_b2.ra_in1k', pretrained=False,
            features_only=True, in_chans=in_channels, out_indices=_EFF_OUT_IDX)
        self.proj = nn.ModuleList([_ConvBnRelu(c, embed_dim) for c in _EFF_CHANNELS])
        self.fuse = nn.Sequential(
            _ConvBnRelu(embed_dim * 4, embed_dim),
            _ConvBnRelu(embed_dim, embed_dim // 2, k=3, p=1))
        self.head = nn.Conv2d(embed_dim // 2, 1, kernel_size=1)

    def forward(self, before, after):
        H4, W4 = before.shape[2] // 4, before.shape[3] // 4
        feats = []
        for i, (fb, fa) in enumerate(zip(self.encoder(before), self.encoder(after))):
            d = self.proj[i](torch.abs(fb - fa))
            feats.append(F.interpolate(d, (H4, W4), mode='bilinear', align_corners=False))
        logits = self.head(self.fuse(torch.cat(feats, dim=1)))
        return F.interpolate(logits, before.shape[2:], mode='bilinear', align_corners=False)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 8 · OSCDDataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

_BANDS = ['B02', 'B03', 'B04', 'B08']   # Blue, Green, Red, NIR  (10 m/px)


def _load_bands(img_dir: Path, bands: list) -> np.ndarray:
    """Load per-band GeoTIFFs → (H, W, C) float32. Handles B02.tif or *_B02.tif naming."""
    arrs = []
    for b in bands:
        tif = img_dir / f'{b}.tif'
        if not tif.exists():
            matches = sorted(img_dir.glob(f'*_{b}.tif'))
            if not matches:
                avail = sorted(p.name for p in img_dir.glob('*.tif'))[:5]
                raise FileNotFoundError(
                    f'No file matching {b}.tif or *_{b}.tif in {img_dir}\n  Available: {avail}')
            tif = matches[0]
        with rasterio.open(tif) as src:
            arrs.append(src.read(1).astype(np.float32))
    return np.stack(arrs, axis=-1)


def _load_mask(path: Path) -> np.ndarray:
    """
    Load OSCD change mask → (H, W) float32 binary {0=no-change, 1=change}.

    This Kaggle dataset (soumikrakshit/onera-satellite-change-detection-dataset)
    uses a 3-value encoding:
      1 = no-change  (~96-99% of pixels)
      2 = change     (~1-4% of pixels)
    Correct binarisation: cm > 1  (i.e. cm == 2).

    Auto-detect fallback handles other OSCD re-packagings automatically.
    """
    with rasterio.open(path) as src:
        cm = src.read(1).astype(np.float32)

    for mask_bool in [cm > 1, cm > 0, cm == 0, cm > 127]:
        if 0.3 <= mask_bool.mean() * 100 <= 85.0:
            return mask_bool.astype(np.float32)

    uv, uc = np.unique(cm, return_counts=True)
    raise ValueError(
        f'Cannot determine mask convention for {path}\n'
        f'  Values: {dict(zip(uv.astype(int).tolist(), uc.tolist()))}')


def _norm_pair(before, after):
    """Joint percentile normalisation [p2, p98] → [0,1], per channel."""
    out_b = np.empty_like(before, dtype=np.float32)
    out_a = np.empty_like(after,  dtype=np.float32)
    for c in range(before.shape[-1]):
        b_ch, a_ch = before[..., c], after[..., c]
        joint = np.concatenate([b_ch.ravel(), a_ch.ravel()])
        lo, hi = np.percentile(joint, 2), np.percentile(joint, 98)
        d = hi - lo
        if d < 1e-8:
            out_b[..., c] = out_a[..., c] = 0.0
        else:
            out_b[..., c] = np.clip((b_ch - lo) / d, 0, 1)
            out_a[..., c] = np.clip((a_ch - lo) / d, 0, 1)
    return out_b, out_a


class OSCDDataset(Dataset):
    """Random-crop dataset for OSCD Sentinel-2 city pairs. Caches images in RAM."""
    def __init__(self, images_dir, labels_dir, cities,
                 patch_size=256, patches_per_city=50, augment=True):
        self.ps, self.ppc, self.aug = patch_size, patches_per_city, augment
        images_dir, labels_dir = Path(images_dir), Path(labels_dir)
        self._triples = []
        for city in cities:
            cd = images_dir / city
            if not cd.is_dir(): continue
            t1 = _find_sub(cd, _T1_OPTS)
            t2 = _find_sub(cd, _T2_OPTS)
            if t1 is None or t2 is None: continue
            md = labels_dir / city / 'cm'
            mp = md / f'{city}-cm.tif'
            if not mp.exists(): mp = md / 'cm.tif'
            if not mp.exists(): continue
            self._triples.append((t1, t2, mp))
        if not self._triples:
            raise ValueError(f'No valid city pairs.\n  images_dir={images_dir}\n  cities={cities}')
        print(f'[OSCDDataset] {len(self._triples)} cities  ps={patch_size}  '
              f'ppc={patches_per_city}  aug={augment}  → {len(self)} samples/epoch')
        self._cache = {}

    def _city(self, i):
        if i not in self._cache:
            t1, t2, mp = self._triples[i]
            b, a = _norm_pair(_load_bands(t1, _BANDS), _load_bands(t2, _BANDS))
            self._cache[i] = (b, a, _load_mask(mp))
        return self._cache[i]

    def preload_all(self):
        print('Preloading:', end=' ', flush=True)
        for i in range(len(self._triples)):
            self._city(i)
            print(self._triples[i][0].parent.name, end=' ', flush=True)
        print('✓')

    def __len__(self): return len(self._triples) * self.ppc

    def __getitem__(self, idx):
        before, after, mask = self._city(idx % len(self._triples))
        H, W = before.shape[:2]; ps = self.ps
        if H < ps or W < ps:
            ph, pw = max(0, ps-H), max(0, ps-W)
            before = np.pad(before, ((0,ph),(0,pw),(0,0)), 'reflect')
            after  = np.pad(after,  ((0,ph),(0,pw),(0,0)), 'reflect')
            mask   = np.pad(mask,   ((0,ph),(0,pw)),       'reflect')
            H, W   = before.shape[:2]
        i = np.random.randint(0, H-ps+1); j = np.random.randint(0, W-ps+1)
        b = before[i:i+ps, j:j+ps].copy()
        a = after [i:i+ps, j:j+ps].copy()
        m = mask  [i:i+ps, j:j+ps].copy()
        if self.aug:
            if np.random.rand() > .5: b,a,m = b[:,::-1].copy(),a[:,::-1].copy(),m[:,::-1].copy()
            if np.random.rand() > .5: b,a,m = b[::-1].copy(),  a[::-1].copy(),  m[::-1].copy()
            k = np.random.randint(0,4)
            if k: b,a,m = np.rot90(b,k).copy(),np.rot90(a,k).copy(),np.rot90(m,k).copy()
        return (torch.from_numpy(b.transpose(2,0,1)),
                torch.from_numpy(a.transpose(2,0,1)),
                torch.from_numpy(m).unsqueeze(0).float())


print('OSCDDataset defined ✓')

## 9 · Train / val split using only valid cities

In [ ]:
# Prefer these canonical OSCD train cities for validation
_PREF_VAL = ['nantes', 'mumbai', 'bordeaux', 'bercy', 'montpellier']
VAL_CITIES   = [c for c in _PREF_VAL   if c in GOOD_CITIES][:3]   # up to 3 val cities
TRAIN_CITIES = [c for c in GOOD_CITIES if c not in VAL_CITIES]

# Fallback: if no preferred val cities survived, take last 3 good cities
if not VAL_CITIES:
    VAL_CITIES   = GOOD_CITIES[-min(3, len(GOOD_CITIES)//3):]
    TRAIN_CITIES = [c for c in GOOD_CITIES if c not in VAL_CITIES]

print(f'Train ({len(TRAIN_CITIES)}): {TRAIN_CITIES}')
print(f'Val   ({len(VAL_CITIES)}):   {VAL_CITIES}')

train_ds = OSCDDataset(images_dir, labels_dir, TRAIN_CITIES,
                       patch_size=PATCH_SIZE, patches_per_city=PATCHES_PER_CITY, augment=True)
val_ds   = OSCDDataset(images_dir, labels_dir, VAL_CITIES,
                       patch_size=PATCH_SIZE, patches_per_city=30, augment=False)
train_ds.preload_all()
val_ds.preload_all()

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'\nTrain: {len(train_ds)} samples  ({len(train_loader)} batches/epoch)')
print(f'Val  : {len(val_ds)} samples  ({len(val_loader)} batches/epoch)')

## 10 · Load model from LEVIR-CD checkpoint

In [ ]:
model = ChangeFormer(in_channels=4).to(device)

state = torch.load(CKPT_LEVIR, map_location=device, weights_only=True)
sd = state['model'] if isinstance(state, dict) and 'model' in state else state
prev_epoch = state.get('epoch', 0) if isinstance(state, dict) else 0
prev_f1    = state.get('best_f1', 0.0) if isinstance(state, dict) else 0.0

missing, unexpected = model.load_state_dict(sd, strict=False)
print(f'Loaded LEVIR checkpoint  epoch={prev_epoch}  best_F1={prev_f1:.4f}')
if missing:    print(f'  Missing keys   : {missing[:3]}')
if unexpected: print(f'  Unexpected keys: {unexpected[:3]}')
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

## 11 · Loss + validation helper

In [ ]:
def dice_loss(logits, targets, eps=1e-6):
    p = torch.sigmoid(logits).reshape(-1)
    t = targets.reshape(-1)
    return 1.0 - (2.0*(p*t).sum() + eps) / (p.sum() + t.sum() + eps)


def combined_loss(logits, targets, pos_weight):
    """0.5 × weighted-BCE + 0.5 × Dice."""
    pw  = torch.tensor([pos_weight], device=logits.device)
    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw)
    return 0.5 * bce + 0.5 * dice_loss(logits, targets)


@torch.no_grad()
def validate(model, loader, threshold=0.35):
    model.eval()
    total_loss = tp = fp = fn = 0
    for before, after, labels in loader:
        before, after, labels = before.to(device), after.to(device), labels.to(device)
        logits = model(before, after)
        total_loss += combined_loss(logits, labels, POS_WEIGHT).item()
        preds = (torch.sigmoid(logits) > threshold).long()
        tgts  = labels.long()
        tp += int((preds &  tgts).sum())
        fp += int((preds & ~tgts).sum())
        fn += int((~preds & tgts).sum())
    model.train()
    eps = 1e-7
    pr = tp/(tp+fp+eps); rc = tp/(tp+fn+eps)
    return {'loss': total_loss/max(len(loader),1),
            'f1': 2*pr*rc/(pr+rc+eps),
            'iou': tp/(tp+fp+fn+eps),
            'prec': pr, 'recall': rc}


print('Loss helpers defined ✓')

## 12 · Fine-tuning loop

In [ ]:
import math, time

VAL_EVERY = 10
optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=1e-4)

def _lr_lambda(ep):
    if ep < WARMUP_EPOCHS: return (ep+1) / max(WARMUP_EPOCHS, 1)
    t = (ep - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * t))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)
best_f1   = 0.0
n_batches = len(train_loader)

print(f'Fine-tuning  epochs={EPOCHS}  LR={PEAK_LR}  pos_weight={POS_WEIGHT}  '
      f'patch={PATCH_SIZE}  train_cities={len(TRAIN_CITIES)}')
print(f'{"Ep":>4}  {"Loss":>8}  {"F1":>7}  {"IoU":>7}  '
      f'{"Prec":>7}  {"Rec":>7}  {"LR":>9}  {"s/ep":>5}')
print('─' * 65)

model.train()
for epoch in range(1, EPOCHS + 1):
    t0 = time.time(); epoch_loss = 0.0
    for before, after, labels in train_loader:
        before = before.to(device, non_blocking=True)
        after  = after .to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = combined_loss(model(before, after), labels, POS_WEIGHT)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    avg = epoch_loss / n_batches
    lr  = optimizer.param_groups[0]['lr']
    el  = time.time() - t0

    if epoch % VAL_EVERY == 0 or epoch == EPOCHS:
        m    = validate(model, val_loader)
        flag = ''
        if m['f1'] > best_f1:
            best_f1 = m['f1']
            torch.save({'model': model.state_dict(), 'epoch': epoch,
                        'best_f1': best_f1, 'train_cities': TRAIN_CITIES,
                        'val_cities': VAL_CITIES}, CKPT_OSCD)
            flag = '  ← saved'
        print(f'{epoch:>4d}  {avg:>8.4f}  {m["f1"]:>7.4f}  {m["iou"]:>7.4f}  '
              f'{m["prec"]:>7.4f}  {m["recall"]:>7.4f}  {lr:>9.2e}  {el:>4.0f}s{flag}')
    else:
        print(f'{epoch:>4d}  {avg:>8.4f}  {"":>49}  {lr:>9.2e}  {el:>4.0f}s')

print('─' * 65)
print(f'Done.  Best val F1 = {best_f1:.4f}')

## 13 · Threshold sweep

Sentinel-2 at 10 m/px produces weaker model activations than LEVIR-CD  
(0.5 m/px → 10 m/px is a 20× resolution gap).  
Optimal threshold is usually **0.05–0.20** after OSCD fine-tuning.

In [ ]:
# Reload best checkpoint
state = torch.load(CKPT_OSCD, map_location=device, weights_only=True)
model.load_state_dict(state['model'])
print(f'Reloaded best checkpoint  epoch={state["epoch"]}  F1={state["best_f1"]:.4f}')

# Collect val probabilities
print('Collecting val probabilities …')
all_proba, all_labels = [], []
model.eval()
with torch.no_grad():
    for before, after, labels in val_loader:
        before, after = before.to(device), after.to(device)
        all_proba.append(torch.sigmoid(model(before, after)).cpu())
        all_labels.append(labels)
model.train()

P = torch.cat(all_proba)
L = torch.cat(all_labels)

print(f'\n{"Thresh":>7}  {"F1":>7}  {"IoU":>7}  {"Prec":>7}  {"Recall":>7}')
print('─' * 48)

best_f1_sw, best_thr = 0.0, 0.10
eps = 1e-7
for thr in [t/100 for t in range(3, 55, 3)]:
    preds = (P > thr).long(); tgts = L.long()
    tp = int((preds &  tgts).sum())
    fp = int((preds & ~tgts).sum())
    fn = int((~preds & tgts).sum())
    pr = tp/(tp+fp+eps); rc = tp/(tp+fn+eps)
    f1 = 2*pr*rc/(pr+rc+eps)
    iou = tp/(tp+fp+fn+eps)
    mk = '  ←' if f1 > best_f1_sw else ''
    print(f'  {thr:.2f}   {f1:.4f}  {iou:.4f}  {pr:.4f}  {rc:.4f}{mk}')
    if f1 > best_f1_sw: best_f1_sw, best_thr = f1, thr

print('─' * 48)
print(f'\nBest F1={best_f1_sw:.4f}  threshold={best_thr:.2f}')
print(f'→ Set  confidence_threshold = {best_thr:.2f}  in app/core/config.py')

## 14 · Save calibrated checkpoint + upload to GCS

In [ ]:
# Bake calibrated threshold into checkpoint metadata
state = torch.load(CKPT_OSCD, map_location='cpu', weights_only=True)
state['confidence_threshold'] = best_thr
state['val_f1']               = best_f1_sw
torch.save(state, CKPT_OSCD)
print(f'Checkpoint updated: confidence_threshold={best_thr:.2f}  F1={best_f1_sw:.4f}')

In [ ]:
print(f'Uploading to gs://{GCS_BUCKET}/{GCS_CKPT_OUT} …')
!gcloud storage cp {CKPT_OSCD} gs://{GCS_BUCKET}/{GCS_CKPT_OUT}
print('Upload complete ✓')
print()
print('═' * 58)
print('NEXT STEPS — in app/core/config.py :')
print('═' * 58)
print(f'  model_checkpoint_path = "checkpoints/changeformer_oscd.pth"')
print(f'  confidence_threshold  = {best_thr:.2f}')
print()
print('Then rebuild Cloud Run:')
print('  gcloud builds submit --config infra/cloudbuild.yaml .')
print('═' * 58)

---
## Appendix · Visualise one val city

Shows: Before (RGB) / After (RGB) / Probability map / Predictions (red) vs GT (green).  
A **healthy** result has:
- `max_proba` between 0.3 and 0.9 (not 0.0009 = old LEVIR-only model)
- `GT change fraction` between 0.3% and 50% (not 100% = degenerate mask)
- Visible red blobs that roughly align with green GT regions

In [ ]:
import matplotlib.pyplot as plt

INSPECT_CITY = VAL_CITIES[0] if VAL_CITIES else TRAIN_CITIES[0]
city_idx = next((i for i,(t1,_,_) in enumerate(val_ds._triples)
                 if INSPECT_CITY in str(t1)), 0)

before, after, mask = val_ds._city(city_idx)
H, W = before.shape[:2]
ci, cj = (H-256)//2, (W-256)//2
b_c = before[ci:ci+256, cj:cj+256]
a_c = after [ci:ci+256, cj:cj+256]
m_c = mask  [ci:ci+256, cj:cj+256]

model.eval()
with torch.no_grad():
    bt = torch.from_numpy(b_c.transpose(2,0,1)).unsqueeze(0).to(device)
    at = torch.from_numpy(a_c.transpose(2,0,1)).unsqueeze(0).to(device)
    prob = torch.sigmoid(model(bt, at)).squeeze().cpu().numpy()
pred = (prob > best_thr).astype(np.uint8)

# B,G,R,NIR → R,G,B for display
rgb_b = np.stack([b_c[...,2], b_c[...,1], b_c[...,0]], axis=-1).clip(0,1)
rgb_a = np.stack([a_c[...,2], a_c[...,1], a_c[...,0]], axis=-1).clip(0,1)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(rgb_b);                         axes[0].set_title('Before (RGB)')
axes[1].imshow(rgb_a);                         axes[1].set_title('After  (RGB)')
axes[2].imshow(prob, cmap='hot', vmin=0, vmax=1); axes[2].set_title('Probability map')
axes[3].imshow(pred, cmap='Reds',   alpha=1.0, vmin=0, vmax=1)
axes[3].imshow(m_c,  cmap='Greens', alpha=0.45, vmin=0, vmax=1)
axes[3].set_title(f'Pred (red) + GT (green)  thr={best_thr:.2f}')
for ax in axes: ax.axis('off')
city_name = val_ds._triples[city_idx][0].parent.name
plt.suptitle(f'City: {city_name}  max_proba={prob.max():.4f}  '
             f'change_frac_pred={pred.mean():.3f}  change_frac_gt={m_c.mean():.3f}')
plt.tight_layout()
plt.savefig('/content/oscd_result.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'GT change fraction : {m_c.mean():.4f}  (should be 0.003 – 0.50)')
print(f'Pred change frac   : {pred.mean():.4f}')
print(f'Max probability    : {prob.max():.4f}  (should be > 0.10)')